# Music Success Analysis - Part 4: Artist and Playlist Analysis

This notebook focuses on analyzing top artists and the impact of playlist inclusion on streaming performance.

## Loading Libraries and Data

First, let's import the necessary libraries and load our cleaned dataset.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings

# Set plotting style and ignore warnings
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

# Display settings for better visualization
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load the cleaned dataset from pickle file
try:
    df = pd.read_pickle('cleaned_music_data.pkl')
    print("Loaded cleaned data from pickle file.")
except FileNotFoundError:
    print("Cleaned data file not found. Please run the '1_Data_Loading_Cleaning.ipynb' notebook first.")
    # If pickle file not found, load from CSV as fallback
    file_path = r"C:\Users\Adilf\Downloads\Most Streamed Spotify Songs 2024.csv (1)\Most Streamed Spotify Songs 2024.csv"
    df = pd.read_csv(file_path, encoding='latin1')
    print("Loaded original data from CSV file as fallback.")

# Display the first few rows of the dataset
df.head()

## Top Artists Analysis

Let's analyze the top artists by streaming numbers and their cross-platform presence.

In [ ]:
# Bar chart of top artists by Spotify streams
# Group by artist and calculate total streams
top_artists = df.groupby('Artist')['Spotify Streams'].sum().sort_values(ascending=False).head(10)

# Create the bar chart
plt.figure(figsize=(14, 8))
ax = top_artists.plot(kind='bar', color='lightgreen')
plt.title('Top 10 Artists by Total Spotify Streams', fontsize=16)
plt.xlabel('Artist', fontsize=12)
plt.ylabel('Total Spotify Streams', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on top of each bar
for i, v in enumerate(top_artists):
    ax.text(i, v + 0.05 * top_artists.max(), f'{v:,.0f}', 
            ha='center', va='bottom', fontsize=10, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart of top artists by YouTube views
# Group by artist and calculate total views
top_youtube_artists = df.groupby('Artist')['YouTube Views'].sum().sort_values(ascending=False).head(10)

# Create the bar chart
plt.figure(figsize=(14, 8))
ax = top_youtube_artists.plot(kind='bar', color='red')
plt.title('Top 10 Artists by Total YouTube Views', fontsize=16)
plt.xlabel('Artist', fontsize=12)
plt.ylabel('Total YouTube Views', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on top of each bar
for i, v in enumerate(top_youtube_artists):
    ax.text(i, v + 0.05 * top_youtube_artists.max(), f'{v:,.0f}', 
            ha='center', va='bottom', fontsize=10, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart of top artists by TikTok views
# Group by artist and calculate total views
top_tiktok_artists = df.groupby('Artist')['TikTok Views'].sum().sort_values(ascending=False).head(10)

# Create the bar chart
plt.figure(figsize=(14, 8))
ax = top_tiktok_artists.plot(kind='bar', color='purple')
plt.title('Top 10 Artists by Total TikTok Views', fontsize=16)
plt.xlabel('Artist', fontsize=12)
plt.ylabel('Total TikTok Views', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on top of each bar
for i, v in enumerate(top_tiktok_artists):
    ax.text(i, v + 0.05 * top_tiktok_artists.max(), f'{v:,.0f}', 
            ha='center', va='bottom', fontsize=10, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Cross-platform artist presence analysis
# Define the platforms to consider
platforms = ['Spotify Streams', 'YouTube Views', 'TikTok Views', 'Pandora Streams', 'Soundcloud Streams']

# Create a function to calculate cross-platform score
def calculate_cross_platform_score(group):
    # Normalize each platform's values (0-1 scale)
    normalized = {}
    for platform in platforms:
        if platform in group and not group[platform].isnull().all():
            max_val = df[platform].max()
            normalized[platform] = group[platform].sum() / max_val
        else:
            normalized[platform] = 0
    
    # Calculate average normalized score across platforms
    cross_platform_score = sum(normalized.values()) / len(platforms)
    
    # Calculate platform presence (percentage of platforms where artist has data)
    platform_presence = sum(1 for p in normalized.values() if p > 0) / len(platforms)
    
    return pd.Series({
        'Cross_Platform_Score': cross_platform_score,
        'Platform_Presence': platform_presence,
        'Spotify_Score': normalized.get('Spotify Streams', 0),
        'YouTube_Score': normalized.get('YouTube Views', 0),
        'TikTok_Score': normalized.get('TikTok Views', 0),
        'Pandora_Score': normalized.get('Pandora Streams', 0),
        'Soundcloud_Score': normalized.get('Soundcloud Streams', 0)
    })

# Group by artist and calculate cross-platform scores
artist_platform_scores = df.groupby('Artist')[platforms].apply(calculate_cross_platform_score)

# Sort by cross-platform score
top_cross_platform_artists = artist_platform_scores.sort_values('Cross_Platform_Score', ascending=False).head(10)
print("Top 10 Artists with Strongest Cross-Platform Presence:")
print(top_cross_platform_artists[['Cross_Platform_Score', 'Platform_Presence']])

In [ ]:
# Create a radar chart for the top 5 artists with strongest cross-platform presence
top_5_artists = top_cross_platform_artists.head(5)
platform_cols = ['Spotify_Score', 'YouTube_Score', 'TikTok_Score', 'Pandora_Score', 'Soundcloud_Score']
platform_names = ['Spotify', 'YouTube', 'TikTok', 'Pandora', 'Soundcloud']

# Create a figure with subplots
fig = plt.figure(figsize=(15, 10))

# Get artist names for the top 5
top_5_artist_names = top_5_artists.index.tolist()

# Create a color palette
colors = plt.cm.tab10(np.linspace(0, 1, 5))

# Number of variables
N = len(platform_cols)

# What will be the angle of each axis in the plot
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # Close the loop

# Create the subplot
ax = plt.subplot(111, polar=True)

# Draw one axis per variable and add labels
plt.xticks(angles[:-1], platform_names, size=12)

# Draw the y-axis labels (0-1)
ax.set_rlabel_position(0)
plt.yticks([0.2, 0.4, 0.6, 0.8], ['0.2', '0.4', '0.6', '0.8'], color='grey', size=10)
plt.ylim(0, 1)

# Plot each artist
for i, artist in enumerate(top_5_artist_names):
    values = top_5_artists.loc[artist, platform_cols].tolist()
    values += values[:1]  # Close the loop
    
    # Plot values
    ax.plot(angles, values, linewidth=2, linestyle='solid', color=colors[i], label=artist)
    ax.fill(angles, values, color=colors[i], alpha=0.1)

# Add legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.title('Top 5 Artists with Strongest Cross-Platform Presence', size=15, y=1.1)

plt.tight_layout()
plt.show()

## Playlist Impact Analysis

Let's analyze the impact of playlist inclusion on streaming numbers.

In [ ]:
# Visualization of playlist impact
# Create bins for playlist count
df['Playlist_Bins'] = pd.cut(df['Spotify Playlist Count'], 
                            bins=[0, 10000, 20000, 30000, 40000, 50000, 100000],
                            labels=['0-10K', '10K-20K', '20K-30K', '30K-40K', '40K-50K', '50K+'])

# Group by playlist bins and calculate average streams
playlist_impact = df.groupby('Playlist_Bins')['Spotify Streams'].mean().reset_index()

# Create the visualization
plt.figure(figsize=(12, 8))
sns.barplot(x='Playlist_Bins', y='Spotify Streams', data=playlist_impact, palette='viridis')
plt.title('Impact of Playlist Inclusion on Average Spotify Streams', fontsize=16)
plt.xlabel('Number of Spotify Playlists', fontsize=12)
plt.ylabel('Average Spotify Streams', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ticklabel_format(style='plain', axis='y')
plt.tight_layout()
plt.show()

# Calculate correlation between playlist count and streams
playlist_correlation = df['Spotify Playlist Count'].corr(df['Spotify Streams'])
print(f"Correlation between Playlist Count and Streams: {playlist_correlation:.4f}")

In [ ]:
# Scatter plot with regression line for playlist impact
plt.figure(figsize=(12, 8))
sns.regplot(x='Spotify Playlist Count', y='Spotify Streams', data=df, 
           scatter_kws={'alpha':0.5, 's':50}, line_kws={'color':'red'})

plt.title('Impact of Playlist Inclusion on Spotify Streams', fontsize=16)
plt.xlabel('Number of Spotify Playlists', fontsize=12)
plt.ylabel('Spotify Streams', fontsize=12)
plt.grid(linestyle='--', alpha=0.7)
plt.ticklabel_format(style='plain', axis='both')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze the relationship between playlist reach and streams
plt.figure(figsize=(12, 8))
sns.regplot(x='Spotify Playlist Reach', y='Spotify Streams', data=df, 
           scatter_kws={'alpha':0.5, 's':50, 'color':'green'}, line_kws={'color':'darkgreen'})

plt.title('Impact of Playlist Reach on Spotify Streams', fontsize=16)
plt.xlabel('Spotify Playlist Reach', fontsize=12)
plt.ylabel('Spotify Streams', fontsize=12)
plt.grid(linestyle='--', alpha=0.7)
plt.ticklabel_format(style='plain', axis='both')
plt.tight_layout()
plt.show()

# Calculate correlation between playlist reach and streams
reach_correlation = df['Spotify Playlist Reach'].corr(df['Spotify Streams'])
print(f"Correlation between Playlist Reach and Streams: {reach_correlation:.4f}")

In [ ]:
# Analyze the efficiency of playlist inclusion (streams per playlist)
df['Streams_Per_Playlist'] = df['Spotify Streams'] / df['Spotify Playlist Count']

# Group by artist and calculate average streams per playlist
artist_efficiency = df.groupby('Artist')['Streams_Per_Playlist'].mean().sort_values(ascending=False).head(10)

# Create the bar chart
plt.figure(figsize=(14, 8))
ax = artist_efficiency.plot(kind='bar', color='orange')
plt.title('Top 10 Artists by Playlist Efficiency (Streams per Playlist)', fontsize=16)
plt.xlabel('Artist', fontsize=12)
plt.ylabel('Average Streams per Playlist', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ticklabel_format(style='plain', axis='y')
plt.tight_layout()
plt.show()